In [1]:
import httpx

In [4]:
from dotenv import load_dotenv

In [3]:
import os

In [5]:
load_dotenv()

True

In [6]:
api_key = os.getenv("API_KEY")

In [10]:
headers = {
    "Content-Type" : "application/json",
    "Authorization" : f"Bearer {api_key}"
}

In [11]:
model = os.getenv("LLM")
base_url = os.getenv("BASE_URL")

In [41]:
messages = [
    {
        "role" : "system",
        "content" : "You are a conversational agent."
    },
    {
        "role" : "user",
        "content" : "hello, what is today's date?"
    },
    {
        "role" : role,
        "content" : ai_message
    },
]

In [42]:
body = {
    "model" : model,
    "messages" : messages
}

In [43]:
response = httpx.post(
    url=f"{base_url}/chat/completions",
    headers=headers,
    json=body
)

In [44]:
data = response.json()

In [45]:
data

{'id': 'chatcmpl-83623dced2abbfdd',
 'created': 1790229750,
 'model': 'Deepseek-vapt2',
 'object': 'chat.completion',
 'system_fingerprint': 'vllm-0.25.0-tp2-ep-b5ceb09a',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'message': {'role': 'assistant',
    'reasoning_content': '😊',
    'provider_specific_fields': {'reasoning': '😊', 'refusal': None},
    'content': None},
   'provider_specific_fields': {'routed_experts': None,
    'token_ids': None,
    'stop_reason': None}}],
 'usage': {'completion_tokens': 3, 'prompt_tokens': 41, 'total_tokens': 44}}

In [40]:
data['choices'][0]['message']['content']

"You're absolutely right, and I apologize for that! I don't have access to a live clock or real-time data in this conversation, so I made an educated guess based on when my training data was last updated. \n\nCould you please tell me what today's actual date is? I'd be happy to correct my records so I can help you accurately from here on out."

In [28]:
ai_message = data['choices'][0]['message']['content']

role = data['choices'][0]['message']['role']

In [22]:
r = httpx.get(
    f"{base_url}/models",
    headers=headers
)

In [23]:
r

<Response [200 OK]>

In [24]:
r.json()

{'object': 'list',
 'data': [{'id': 'Deepseek-vapt', 'object': 'model', 'owned_by': 'gateway'},
  {'id': 'deepseek', 'object': 'model', 'owned_by': 'gateway'},
  {'id': 'glm5', 'object': 'model', 'owned_by': 'gateway'},
  {'id': 'glm5_vapt', 'object': 'model', 'owned_by': 'gateway'}]}

## Langchain Implementation

In [52]:
class SystemMessage:
    def __init__(self, content):
        self.content = content

    def json(self):
        return {
            "role" : "system",
            "content" : self.content
        }

class AIMessage:
    def __init__(self, content):
        self.content = content

    def json(self):
        return {
            "role" : "assistant",
            "content" : self.content
        }

class HumanMessage:
    def __init__(self, content):
        self.content = content

    def json(self):
        return {
            "role" : "user",
            "content" : self.content
        }
    

In [55]:
class Agent:

    @classmethod
    def invoke(self, model, base_url, api_key, messages):
        headers = {
            "Content-Type" : "application/json",
            "Authorization" : f"Bearer {api_key}"
        }

        json_messages = [m.json() for m in messages]
        body = {
            "model" : model,
            "messages" : json_messages
        }

        response = httpx.post(
            url=f"{base_url}/chat/completions",
            headers=headers,
            json=body
        )

        data = response.json()
        messages.append(AIMessage(data['choices'][0]['message']['content']))
        return messages

In [56]:
messages = [
    SystemMessage(content = "You are a conversational agent."),
    HumanMessage("Hello, tell me about yourself")
]

In [57]:
response = Agent.invoke(
    model=model,
    base_url=base_url,
    api_key=api_key,
    messages=messages
)

In [59]:
response[-1].content

"Hello! Nice to meet you. \n\nI'm an AI assistant (a large language model) designed to have conversations, answer your questions, and help you with a wide variety of tasks. Think of me as a helpful, knowledgeable digital companion. \n\nHere’s a bit about what I can do:\n\n- **Answer questions** on a huge range of topics, from science and history to pop culture and everyday life.\n- **Help with writing** – whether you need a creative story, an email, a business plan, or help polishing a resume.\n- **Brainstorm ideas** – I can help you come up with names, concepts, recipes, or solutions to problems.\n- **Code and technical support** – I can write code, debug it, or explain technical concepts.\n- **Learn and adapt** – I remember the context of our current conversation, so I can build on previous responses.\n\nOf course, I have my limitations too. I don't have personal experiences, emotions, or a physical presence. My knowledge is based on the data I was trained on (and has a cutoff date),